# Reconstructing the surasan092 DINOv3-S+ arm (CPU)

Rebuilds an unpublished inference path from `focal_fold0_final.pt` and verifies it against the
author's own published OOF predictions, which are an exact target.

Already solved, without a GPU:
- **backbone**: `DINOv3ViTConfig(..., use_gated_mlp=True)` loads all 235 tensors with
  `missing 0, unexpected 0`. That flag is the "+" in ViT-S/16+; without it 24 `mlp.gate_proj.*`
  tensors are silently dropped by `strict=False`.
- **window pooling**: plain mean over the 10 windows, recovered to `max|err| = 0.0` by comparing
  `val_window_predictions.npy` (10,882,12) against `val_predictions.npy` (882,12).

Open here: the head forward. 12 slices with `window_size 3` gives exactly the 10 published
windows, so window *w* takes slices `w..w+2` from each of the 6 slots, and 12 learned queries
attend over those 6 slot vectors. How `attn_gate` and `slot_prior` enter that attention is the
unknown, so this greps the small space of plausible variants against the exact target.

Backbone features are computed **once** and cached; the variants are then graded instantly.

In [ ]:
import os, json, math, time
import numpy as np, pandas as pd, torch, torch.nn.functional as Fn
torch.set_grad_enabled(False); torch.set_num_threads(os.cpu_count() or 4)

def find(name, root='/kaggle/input'):
    for r,d,f in os.walk(root):
        d[:] = [x for x in d if x not in ('train_series','test_series')]
        if name in f: return os.path.join(r,name)
    raise FileNotFoundError(name)

CK = torch.load(find('focal_fold0_final.pt'), map_location='cpu', weights_only=False)
SD, CFG = CK['model_state'], CK['config']
SLOTS  = list(zip(CFG['slot_names'], CFG['slot_planes'], CFG['slot_kinds'],
                  CFG['slot_windows'], CFG['slot_min_coverage']))
IMG, CROP_MM, NSL, WIN = CFG['image_size'], CFG['crop_mm'], CFG['canonical_slices_per_slot'], CFG['window_size']
NWIN = NSL - WIN + 1
print(f'{len(SLOTS)} slots | img {IMG} | crop {CROP_MM}mm | {NSL} slices | window {WIN} -> {NWIN} windows')
for s in SLOTS: print('  ', s)

from transformers.models.dinov3_vit.configuration_dinov3_vit import DINOv3ViTConfig
from transformers.models.dinov3_vit.modeling_dinov3_vit import DINOv3ViTModel
cfg = DINOv3ViTConfig(hidden_size=384, num_hidden_layers=12, num_attention_heads=6,
                      patch_size=16, num_register_tokens=4, image_size=IMG,
                      intermediate_size=1536, use_gated_mlp=True)
BB = DINOv3ViTModel(cfg).eval()
miss, unexp = BB.load_state_dict({k[9:]: v for k,v in SD.items() if k.startswith('backbone.')}, strict=False)
assert not miss and not unexp, (len(miss), len(unexp))
NPREFIX = 1 + cfg.num_register_tokens
print(f'backbone loaded exactly | prefix tokens {NPREFIX}')

In [ ]:
# Which studies to reproduce: fold-0 validation rows, where the target predictions live.
VIDX = np.load(find('focal_fold0_val_indices.npy'))
VPRD = np.load(find('focal_fold0_val_predictions.npy'))
VWIN = np.load(find('focal_fold0_val_window_predictions.npy'))
print('fold-0 val:', VIDX.shape, VPRD.shape, VWIN.shape)

COMP = os.path.dirname(find('train.csv'))
tr   = pd.read_csv(os.path.join(COMP,'train.csv'))
ser  = pd.read_csv(os.path.join(COMP,'train_series.csv'))
N_STUDY = int(os.environ.get('N_STUDY', '12'))
sel = VIDX[:N_STUDY]
uids = tr.StudyInstanceUID.astype(str).values[sel]
print(f'reproducing {len(uids)} studies')
TARGET_STUDY  = VPRD[:N_STUDY]
TARGET_WINDOW = VWIN[:, :N_STUDY]

In [ ]:
# Preprocessing, straight from the checkpoint config: pick one series per slot, take NSL slices
# evenly across that slot's band, crop CROP_MM around centre using PixelSpacing, resize to IMG.
import pydicom, cv2
cv2.setNumThreads(1)
ser['plane'] = ser['Anatomical_Plane']
def slot_match(g, plane, kind):
    if kind == 'FLUID_FS':   m = (g.Fluid_Sensitive==1) & (g.Fat_Suppression==1)
    elif kind == 'FLUID_NOFS': m = (g.Fluid_Sensitive==1) & (g.Fat_Suppression==0)
    else:                    m = (g.Fluid_Sensitive==0)
    return g[(g.plane==plane) & m]

def ordered_slices(sdir):
    keyed=[]
    for f in os.listdir(sdir):
        if not f.endswith('.dcm'): continue
        try:
            ds = pydicom.dcmread(os.path.join(sdir,f), stop_before_pixels=True)
            keyed.append((float(getattr(ds,'InstanceNumber',0)), os.path.join(sdir,f)))
        except Exception: pass
    return [p for _,p in sorted(keyed)]

def render(path):
    ds = pydicom.dcmread(path); a = ds.pixel_array.astype(np.float32)
    try: ps = float(ds.PixelSpacing[0])
    except Exception: ps = CROP_MM/max(a.shape)
    half = int(round(CROP_MM/ps/2)); cy,cx = a.shape[0]//2, a.shape[1]//2
    c = a[max(0,cy-half):cy+half, max(0,cx-half):cx+half]
    if c.size==0: return None
    lo,hi = np.percentile(c[::4,::4],[1,99])
    c = np.clip((c-lo)/max(hi-lo,1e-6),0,1)
    return cv2.resize(c,(IMG,IMG),interpolation=cv2.INTER_AREA)

def build_study(uid):
    g = ser[ser.StudyInstanceUID==uid]
    out  = np.zeros((len(SLOTS), NSL, IMG, IMG), np.float32)
    mask = np.zeros(len(SLOTS), np.float32)
    for si,(nm,plane,kind,band,mincov) in enumerate(SLOTS):
        cand = slot_match(g, plane, kind)
        if not len(cand): continue
        sdir = os.path.join(COMP,'train_series',uid,str(cand.iloc[0].SeriesInstanceUID))
        if not os.path.isdir(sdir): continue
        files = ordered_slices(sdir)
        if len(files) < 3: continue
        lo,hi = band
        pick = np.unique(np.linspace(int(lo*(len(files)-1)), int(hi*(len(files)-1)), NSL).astype(int))
        while len(pick) < NSL: pick = np.append(pick, pick[-1])
        for k,pi in enumerate(pick[:NSL]):
            im = render(files[pi])
            if im is not None: out[si,k] = im
        mask[si] = 1.0
    return out, mask

t0=time.time(); PIX=[]; MASK=[]
for i,u in enumerate(uids):
    a,m = build_study(u); PIX.append(a); MASK.append(m)
    print(f'  {i+1}/{len(uids)} slots={int(m.sum())}/6  {time.time()-t0:.0f}s', flush=True)
PIX=np.stack(PIX); MASK=np.stack(MASK)
print('pixels',PIX.shape,'| mean slots filled',MASK.mean())

In [ ]:
# Backbone features ONCE: (study, slot, window) -> 1152. Cached so head variants are free.
def feats_for(img3):                      # img3: (B,3,IMG,IMG)
    h = BB(pixel_values=torch.from_numpy(img3)).last_hidden_state
    cls = h[:,0]; pat = h[:,NPREFIX:]
    k = max(1,int(round(CFG['focal_top_fraction']*pat.shape[1])))
    focal = pat.topk(k,dim=1).values.mean(1)
    return torch.cat([cls, pat.mean(1), focal], -1)

S,NS = len(uids), len(SLOTS)
FEAT = np.zeros((S,NS,NWIN,1152), np.float32)
t0=time.time(); done=0; total=S*NS*NWIN
for i in range(S):
    for s in range(NS):
        if MASK[i,s]==0: continue
        batch = np.stack([PIX[i,s,w:w+WIN] for w in range(NWIN)])   # (NWIN,3,IMG,IMG)
        FEAT[i,s] = feats_for(batch).numpy()
        done += NWIN
        print(f'  study {i+1}/{S} slot {s} | {done}/{total} windows | {time.time()-t0:.0f}s', flush=True)
np.save('/kaggle/working/FEAT.npy',FEAT); np.save('/kaggle/working/MASK.npy',MASK)
print('features',FEAT.shape)

In [ ]:
# Head variants. Everything except the attention wiring is pinned by the tensor shapes.
P = {k: SD[k].float() for k in ['slot_queries','attn_gate','slot_prior','classifier_w',
     'classifier_b','feature_proj.weight','feature_proj.bias','slot_embed.weight',
     'slot_norm.weight','slot_norm.bias','study_norm.weight','study_norm.bias']}
D = P['slot_queries'].shape[1]

def head(feat, mask, gate_mode, prior_mode, scale):
    # feat (NS,1152) for ONE window; mask (NS,)
    v = feat @ P['feature_proj.weight'].T + P['feature_proj.bias']      # (NS,D)
    v = v + P['slot_embed.weight']
    v = Fn.layer_norm(v,(D,),P['slot_norm.weight'],P['slot_norm.bias'])
    sc = P['slot_queries'] @ v.T                                        # (12,NS)
    if scale == 'sqrt': sc = sc / math.sqrt(D)
    if prior_mode == 'add':  sc = sc + P['slot_prior']
    if prior_mode == 'mul':  sc = sc * P['slot_prior']
    sc = sc.masked_fill(mask[None,:]==0, float('-inf'))
    w  = torch.softmax(sc, -1)
    ctx = w @ v                                                         # (12,D)
    g = torch.sigmoid(P['attn_gate'])[:,None]
    m = (v*mask[:,None]).sum(0)/mask.sum().clamp(min=1)
    if   gate_mode=='gate_ctx_mean': out = g*ctx + (1-g)*m[None,:]
    elif gate_mode=='gate_ctx':      out = g*ctx
    elif gate_mode=='none':          out = ctx
    out = Fn.layer_norm(out,(D,),P['study_norm.weight'],P['study_norm.bias'])
    return torch.sigmoid((out*P['classifier_w']).sum(-1)+P['classifier_b'])

best=None
for gate_mode in ['gate_ctx_mean','gate_ctx','none']:
  for prior_mode in ['add','none','mul']:
    for scale in ['sqrt','none']:
      W=np.zeros((NWIN,S,12),np.float32)
      for i in range(S):
        mk=torch.from_numpy(MASK[i])
        for w in range(NWIN):
            W[w,i]=head(torch.from_numpy(FEAT[i,:,w]),mk,gate_mode,prior_mode,scale).numpy()
      study=W.mean(0)
      e=np.abs(study-TARGET_STUDY); ew=np.abs(W-TARGET_WINDOW)
      corr=np.corrcoef(study.ravel(),TARGET_STUDY.ravel())[0,1]
      tag=f'{gate_mode}/{prior_mode}/scale={scale}'
      print(f'  {tag:34s} max|err| {e.max():.5f}  mean {e.mean():.5f}  win {ew.mean():.5f}  r={corr:.4f}')
      if best is None or e.mean()<best[1]: best=(tag,e.mean(),corr)
print(f'\nBEST: {best[0]}  mean|err|={best[1]:.6f}  r={best[2]:.4f}')
print('VERDICT:', 'EXACT' if best[1]<1e-4 else ('CLOSE' if best[2]>0.98 else 'NOT REPRODUCED'))